In [1]:
import $ivy.`org.apache.spark::spark-sql:4.1.1`

import org.apache.spark.sql.SparkSession
import org.apache.spark.sql.functions._

val spark = SparkSession.builder()
  .appName("Joins-Ejemplos")
  .master("local[*]")
  .config("spark.sql.shuffle.partitions", "4")
  .config("spark.sql.crossJoin.enabled", "true")
  .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
import spark.implicits._

println(s"Spark ${spark.version} — Scala ${scala.util.Properties.versionString} ✅")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/29 09:30:07 INFO SparkContext: Running Spark version 4.1.1
26/04/29 09:30:07 INFO SparkContext: OS info Windows 11, 10.0, amd64
26/04/29 09:30:07 INFO SparkContext: Java version 17.0.12+8-LTS-286
26/04/29 09:30:07 INFO ResourceUtils: ==============================================================
26/04/29 09:30:07 INFO ResourceUtils: No custom resources configured for spark.driver.
26/04/29 09:30:07 INFO ResourceUtils: ==============================================================
26/04/29 09:30:07 INFO SparkContext: Submitted application: Joins-Ejemplos
26/04/29 09:30:07 INFO SecurityManager: Changing view acls to: Imp_06 - Ma26/04/29 09:30:07 INFO SecurityManager: Changing view acls to: Imp_06 - Ma26/04/29 09:30:07 INFO SecurityManager: Changing view acls to: Imp_06 - Ma26/04/29 09:30:07 INFO SecurityManager: Changing view acls to: Imp_06 - Ma26/04/29 09:30:07 INFO SecurityManager: Changing view acl

2026-04-29T07:30:08.049681400Z scala-kernel-interpreter-1 ERROR An exception occurred processing Appender console org.apache.logging.log4j.core.appender.AppenderLoggingException: java.lang.AssertionError: assertion failed
	at org.apache.logging.log4j.core.config.AppenderControl.tryCallAppender(AppenderControl.java:164)
	at org.apache.logging.log4j.core.config.AppenderControl.callAppender0(AppenderControl.java:133)
	at org.apache.logging.log4j.core.config.AppenderControl.callAppenderPreventRecursion(AppenderControl.java:124)
	at org.apache.logging.log4j.core.config.AppenderControl.callAppender(AppenderControl.java:88)
	at org.apache.logging.log4j.core.config.LoggerConfig.callAppenders(LoggerConfig.java:714)
	at org.apache.logging.log4j.core.config.LoggerConfig.processLogEvent(LoggerConfig.java:672)
	at org.apache.logging.log4j.core.config.LoggerConfig.log(LoggerConfig.java:648)
	at org.apache.logging.log4j.core.config.LoggerConfig.log(LoggerConfig.java:584)
	at org.apache.logging.log4j.

import $ivy.$
import org.apache.spark.sql.SparkSession
import org.apache.spark.sql.functions._
spark: SparkSession = org.apache.spark.sql.classic.SparkSession@55981713
import spark.implicits._

In [2]:
// DataFrame A — clientes
val clientes = Seq(
  (1, "Ana"),
  (2, "Borja"),
  (3, "Carmen")
).toDF("id", "nombre")

// DataFrame B — pedidos
val pedidos = Seq(
  ("P1", 1, 200.0),
  ("P2", 1,  80.0),
  ("P3", 2, 350.0)
).toDF("ped", "clienteId", "importe")

println("=== DataFrame A: clientes ===")
clientes.show()

println("=== DataFrame B: pedidos ===")
pedidos.show()

=== DataFrame A: clientes ===
+---+------+
| id|nombre|
+---+------+
|  1|   Ana|
|  2| Borja|
|  3|Carmen|
+---+------+

=== DataFrame B: pedidos ===
+---+---------+-------+
|ped|clienteId|importe|
+---+---------+-------+
| P1|        1|  200.0|
| P2|        1|   80.0|
| P3|        2|  350.0|
+---+---------+-------+



clientes: org.apache.spark.sql.package.DataFrame = [id: int, nombre: string]
pedidos: org.apache.spark.sql.package.DataFrame = [ped: string, clienteId: int ... 1 more field]

In [3]:


val inner = clientes.join(pedidos, clientes("id") === pedidos("clienteId"), "inner")

println("=== INNER JOIN ===")
inner.show()
println(s"Filas: ${inner.count()}")

=== INNER JOIN ===
+---+------+---+---------+-------+
| id|nombre|ped|clienteId|importe|
+---+------+---+---------+-------+
|  1|   Ana| P1|        1|  200.0|
|  1|   Ana| P2|        1|   80.0|
|  2| Borja| P3|        2|  350.0|
+---+------+---+---------+-------+

Filas: 3


inner: org.apache.spark.sql.package.DataFrame = [id: int, nombre: string ... 3 more fields]

In [4]:
// LEFT JOIN
val left = clientes.join(pedidos, clientes("id") === pedidos("clienteId"), "left")

println("=== LEFT JOIN ===")
left.show()
println(s"Filas: ${left.count()}")

=== LEFT JOIN ===
+---+------+----+---------+-------+
| id|nombre| ped|clienteId|importe|
+---+------+----+---------+-------+
|  1|   Ana|  P2|        1|   80.0|
|  1|   Ana|  P1|        1|  200.0|
|  2| Borja|  P3|        2|  350.0|
|  3|Carmen|NULL|     NULL|   NULL|
+---+------+----+---------+-------+

Filas: 4


left: org.apache.spark.sql.package.DataFrame = [id: int, nombre: string ... 3 more fields]

In [5]:

val right = clientes.join(pedidos, clientes("id") === pedidos("clienteId"), "right")

println("=== RIGHT JOIN ===")
right.show()
println(s"Filas: ${right.count()}")

=== RIGHT JOIN ===
+---+------+---+---------+-------+
| id|nombre|ped|clienteId|importe|
+---+------+---+---------+-------+
|  1|   Ana| P1|        1|  200.0|
|  1|   Ana| P2|        1|   80.0|
|  2| Borja| P3|        2|  350.0|
+---+------+---+---------+-------+

Filas: 3


right: org.apache.spark.sql.package.DataFrame = [id: int, nombre: string ... 3 more fields]

In [6]:

val full = clientes.join(pedidos, clientes("id") === pedidos("clienteId"), "full")

println("=== FULL OUTER JOIN ===")
full.show()
println(s"Filas: ${full.count()}")

=== FULL OUTER JOIN ===
+---+------+----+---------+-------+
| id|nombre| ped|clienteId|importe|
+---+------+----+---------+-------+
|  1|   Ana|  P1|        1|  200.0|
|  1|   Ana|  P2|        1|   80.0|
|  2| Borja|  P3|        2|  350.0|
|  3|Carmen|NULL|     NULL|   NULL|
+---+------+----+---------+-------+

Filas: 4


full: org.apache.spark.sql.package.DataFrame = [id: int, nombre: string ... 3 more fields]

In [7]:
// CROSS JOIN — en Spark 4.x se usa el tipo "cross" dentro de .join()
// Requiere spark.sql.crossJoin.enabled = true
// En Spark 4.x el cross join se expresa con lit(true) como condición y tipo "cross"
// La configuración spark.sql.crossJoin.enabled = true ya está activada en la Celda 1
val cross = clientes.join(pedidos, lit(true), "cross")

println("=== CROSS JOIN ===")
cross.show()
println(s"Filas: ${cross.count()} (${clientes.count()} clientes × ${pedidos.count()} pedidos)")

=== CROSS JOIN ===
+---+------+---+---------+-------+
| id|nombre|ped|clienteId|importe|
+---+------+---+---------+-------+
|  1|   Ana| P1|        1|  200.0|
|  2| Borja| P1|        1|  200.0|
|  3|Carmen| P1|        1|  200.0|
|  1|   Ana| P2|        1|   80.0|
|  2| Borja| P2|        1|   80.0|
|  3|Carmen| P2|        1|   80.0|
|  1|   Ana| P3|        2|  350.0|
|  2| Borja| P3|        2|  350.0|
|  3|Carmen| P3|        2|  350.0|
+---+------+---+---------+-------+

Filas: 9 (3 clientes × 3 pedidos)


cross: org.apache.spark.sql.package.DataFrame = [id: int, nombre: string ... 3 more fields]

In [8]:
// LEFT SEMI JOIN
val semi = clientes.join(pedidos, clientes("id") === pedidos("clienteId"), "semi")

println("=== LEFT SEMI JOIN ===")
semi.show()
println(s"Filas: ${semi.count()}")

=== LEFT SEMI JOIN ===
+---+------+
| id|nombre|
+---+------+
|  1|   Ana|
|  2| Borja|
+---+------+

Filas: 2


semi: org.apache.spark.sql.package.DataFrame = [id: int, nombre: string]

In [9]:
// LEFT ANTI JOIN
val anti = clientes.join(pedidos, clientes("id") === pedidos("clienteId"), "anti")

println("=== LEFT ANTI JOIN ===")
anti.show()
println(s"Filas: ${anti.count()}")

=== LEFT ANTI JOIN ===
+---+------+
| id|nombre|
+---+------+
|  3|Carmen|
+---+------+

Filas: 1


anti: org.apache.spark.sql.package.DataFrame = [id: int, nombre: string]

In [10]:
// DataFrame A — clientes
// Columnas: id, nombre
val clientes = Seq(
  (1, "Ana"),
  (2, "Borja"),
  (3, "Carmen")
).toDF("id", "nombre")

// DataFrame B — pedidos
// Columnas: id, clienteId, importe
// ATENCIÓN: también tiene una columna llamada "id" (el id del pedido)
val pedidos = Seq(
  (101, 1, 200.0),
  (102, 1,  80.0),
  (103, 2, 350.0)
).toDF("id", "clienteId", "importe")

println("=== DataFrame A: clientes ===")
clientes.show()
println("Columnas de clientes: " + clientes.columns.mkString(", "))

println("\n=== DataFrame B: pedidos ===")
pedidos.show()
println("Columnas de pedidos: " + pedidos.columns.mkString(", "))

println("\n⚠️  CONFLICTO: ambos DataFrames tienen una columna llamada 'id'")

=== DataFrame A: clientes ===
+---+------+
| id|nombre|
+---+------+
|  1|   Ana|
|  2| Borja|
|  3|Carmen|
+---+------+

Columnas de clientes: id, nombre

=== DataFrame B: pedidos ===
+---+---------+-------+
| id|clienteId|importe|
+---+---------+-------+
|101|        1|  200.0|
|102|        1|   80.0|
|103|        2|  350.0|
+---+---------+-------+

Columnas de pedidos: id, clienteId, importe

⚠️  CONFLICTO: ambos DataFrames tienen una columna llamada 'id'


clientes: org.apache.spark.sql.package.DataFrame = [id: int, nombre: string]
pedidos: org.apache.spark.sql.package.DataFrame = [id: int, clienteId: int ... 1 more field]

In [11]:
// Hacemos el join sin resolver el conflicto
val resultado = clientes.join(pedidos, clientes("id") === pedidos("clienteId"))

// Inspeccionamos el schema del resultado: veremos DOS columnas llamadas "id"
println("=== Schema tras el join (sin resolver el conflicto) ===")
resultado.printSchema()
println("Columnas del resultado: " + resultado.columns.mkString(", "))
println()

// Mostramos los datos — esto funciona porque show() no selecciona por nombre
println("=== Datos (show funciona porque no selecciona por nombre) ===")
resultado.show()

=== Schema tras el join (sin resolver el conflicto) ===
root
 |-- id: integer (nullable = false)
 |-- nombre: string (nullable = true)
 |-- id: integer (nullable = false)
 |-- clienteId: integer (nullable = false)
 |-- importe: double (nullable = false)

Columnas del resultado: id, nombre, id, clienteId, importe

=== Datos (show funciona porque no selecciona por nombre) ===
+---+------+---+---------+-------+
| id|nombre| id|clienteId|importe|
+---+------+---+---------+-------+
|  1|   Ana|102|        1|   80.0|
|  1|   Ana|101|        1|  200.0|
|  2| Borja|103|        2|  350.0|
+---+------+---+---------+-------+



resultado: org.apache.spark.sql.package.DataFrame = [id: int, nombre: string ... 3 more fields]

In [12]:
// ─────────────────────────────────────────────────
// TABLA GRANDE: pedidos
// En producción tendría millones de filas.
// Aquí usamos 12 filas para que el ejemplo sea legible.
// ─────────────────────────────────────────────────
val pedidos = Seq(
  ("P001", 1, "CAT-A", 120.0),
  ("P002", 1, "CAT-B",  45.0),
  ("P003", 2, "CAT-A", 200.0),
  ("P004", 2, "CAT-C",  89.0),
  ("P005", 3, "CAT-B", 310.0),
  ("P006", 3, "CAT-A",  55.0),
  ("P007", 4, "CAT-C", 175.0),
  ("P008", 4, "CAT-B",  30.0),
  ("P009", 5, "CAT-A", 250.0),
  ("P010", 5, "CAT-C",  99.0),
  ("P011", 6, "CAT-B", 145.0),
  ("P012", 6, "CAT-A",  70.0)
).toDF("pedidoId", "clienteId", "categoriaId", "importe")

// ─────────────────────────────────────────────────
// TABLA PEQUEÑA: categorías
// Solo 3 filas — catálogo estático que raramente cambia.
// Esta es la candidata perfecta para broadcast.
// ─────────────────────────────────────────────────
val categorias = Seq(
  ("CAT-A", "Electrónica",  "Alta gama"),
  ("CAT-B", "Hogar",        "Básica"),
  ("CAT-C", "Deportes",     "Media")
).toDF("id", "nombreCategoria", "segmento")

println("=== Tabla GRANDE: pedidos ===")
pedidos.show()
println(s"Filas en pedidos:    ${pedidos.count()}")

println("\n=== Tabla PEQUEÑA: categorias ===")
categorias.show()
println(s"Filas en categorias: ${categorias.count()}")

println("\n💡 'categorias' es la candidata al broadcast: pocas filas, datos estáticos.")

=== Tabla GRANDE: pedidos ===
+--------+---------+-----------+-------+
|pedidoId|clienteId|categoriaId|importe|
+--------+---------+-----------+-------+
|    P001|        1|      CAT-A|  120.0|
|    P002|        1|      CAT-B|   45.0|
|    P003|        2|      CAT-A|  200.0|
|    P004|        2|      CAT-C|   89.0|
|    P005|        3|      CAT-B|  310.0|
|    P006|        3|      CAT-A|   55.0|
|    P007|        4|      CAT-C|  175.0|
|    P008|        4|      CAT-B|   30.0|
|    P009|        5|      CAT-A|  250.0|
|    P010|        5|      CAT-C|   99.0|
|    P011|        6|      CAT-B|  145.0|
|    P012|        6|      CAT-A|   70.0|
+--------+---------+-----------+-------+

Filas en pedidos:    12

=== Tabla PEQUEÑA: categorias ===
+-----+---------------+---------+
|   id|nombreCategoria| segmento|
+-----+---------------+---------+
|CAT-A|    Electrónica|Alta gama|
|CAT-B|          Hogar|   Básica|
|CAT-C|       Deportes|    Media|
+-----+---------------+---------+

Filas en catego

pedidos: org.apache.spark.sql.package.DataFrame = [pedidoId: string, clienteId: int ... 2 more fields]
categorias: org.apache.spark.sql.package.DataFrame = [id: string, nombreCategoria: string ... 1 more field]

In [13]:
import org.apache.spark.sql.functions.broadcast
// ── Desactivamos el broadcast automático para ver el plan sin optimización ──
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)

val umbralActual = spark.conf.get("spark.sql.autoBroadcastJoinThreshold")
println(s"Umbral autoBroadcastJoinThreshold: $umbralActual")
println("(valor -1 = broadcast automático DESACTIVADO)")
println()
println("Ahora Spark usará SortMergeJoin aunque la tabla sea pequeña.")
println("Esto nos permitirá ver la diferencia en el plan de ejecución.")

Umbral autoBroadcastJoinThreshold: -1
(valor -1 = broadcast automático DESACTIVADO)

Ahora Spark usará SortMergeJoin aunque la tabla sea pequeña.
Esto nos permitirá ver la diferencia en el plan de ejecución.


import org.apache.spark.sql.functions.broadcast
umbralActual: String = "-1"

In [14]:
// JOIN NORMAL — sin broadcast (broadcast automático ya está desactivado)
val sinBroadcast = pedidos.join(
  categorias,
  pedidos("categoriaId") === categorias("id")
)

println("=== Resultado del join sin broadcast ===")
sinBroadcast.select(
  col("pedidoId"),
  col("clienteId"),
  col("nombreCategoria"),
  col("segmento"),
  col("importe")
).orderBy("pedidoId").show()

println("\n=== Plan de ejecución SIN broadcast ===")
println("(busca 'SortMergeJoin' y 'Exchange' en el plan)")
println("─" * 60)
sinBroadcast.explain()

=== Resultado del join sin broadcast ===
+--------+---------+---------------+---------+-------+
|pedidoId|clienteId|nombreCategoria| segmento|importe|
+--------+---------+---------------+---------+-------+
|    P001|        1|    Electrónica|Alta gama|  120.0|
|    P002|        1|          Hogar|   Básica|   45.0|
|    P003|        2|    Electrónica|Alta gama|  200.0|
|    P004|        2|       Deportes|    Media|   89.0|
|    P005|        3|          Hogar|   Básica|  310.0|
|    P006|        3|    Electrónica|Alta gama|   55.0|
|    P007|        4|       Deportes|    Media|  175.0|
|    P008|        4|          Hogar|   Básica|   30.0|
|    P009|        5|    Electrónica|Alta gama|  250.0|
|    P010|        5|       Deportes|    Media|   99.0|
|    P011|        6|          Hogar|   Básica|  145.0|
|    P012|        6|    Electrónica|Alta gama|   70.0|
+--------+---------+---------------+---------+-------+


=== Plan de ejecución SIN broadcast ===
(busca 'SortMergeJoin' y 'Exchange' e

sinBroadcast: org.apache.spark.sql.package.DataFrame = [pedidoId: string, clienteId: int ... 5 more fields]

In [15]:
import org.apache.spark.sql.functions.broadcast
// JOIN CON BROADCAST FORZADO
// broadcast() le indica a Spark que envíe 'categorias' completo a cada nodo
val conBroadcast = pedidos.join(
  broadcast(categorias),            // ← aquí está la clave
  pedidos("categoriaId") === categorias("id")
)

println("=== Resultado del join CON broadcast ===")
conBroadcast.select(
  col("pedidoId"),
  col("clienteId"),
  col("nombreCategoria"),
  col("segmento"),
  col("importe")
).orderBy("pedidoId").show()

println("\n=== Plan de ejecución CON broadcast ===")
println("(busca 'BroadcastHashJoin' y 'BroadcastExchange' en el plan)")
println("─" * 60)
conBroadcast.explain()

=== Resultado del join CON broadcast ===
+--------+---------+---------------+---------+-------+
|pedidoId|clienteId|nombreCategoria| segmento|importe|
+--------+---------+---------------+---------+-------+
|    P001|        1|    Electrónica|Alta gama|  120.0|
|    P002|        1|          Hogar|   Básica|   45.0|
|    P003|        2|    Electrónica|Alta gama|  200.0|
|    P004|        2|       Deportes|    Media|   89.0|
|    P005|        3|          Hogar|   Básica|  310.0|
|    P006|        3|    Electrónica|Alta gama|   55.0|
|    P007|        4|       Deportes|    Media|  175.0|
|    P008|        4|          Hogar|   Básica|   30.0|
|    P009|        5|    Electrónica|Alta gama|  250.0|
|    P010|        5|       Deportes|    Media|   99.0|
|    P011|        6|          Hogar|   Básica|  145.0|
|    P012|        6|    Electrónica|Alta gama|   70.0|
+--------+---------+---------------+---------+-------+


=== Plan de ejecución CON broadcast ===
(busca 'BroadcastHashJoin' y 'Broadca

import org.apache.spark.sql.functions.broadcast
conBroadcast: org.apache.spark.sql.package.DataFrame = [pedidoId: string, clienteId: int ... 5 more fields]

In [16]:
import org.apache.spark.sql.functions.broadcast
// ── Activamos el broadcast automático con un umbral de 10 MB ──
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", 10 * 1024 * 1024)

val umbralActivo = spark.conf.get("spark.sql.autoBroadcastJoinThreshold")
println(s"Umbral autoBroadcastJoinThreshold: $umbralActivo bytes (${10} MB)")
println()

// Ahora hacemos el join SIN broadcast() explícito
// Spark debería aplicarlo automáticamente porque 'categorias' es pequeña
val autobroadcast = pedidos.join(
  categorias,                                       // sin broadcast() explícito
  pedidos("categoriaId") === categorias("id")
)

println("=== Plan con broadcast AUTOMÁTICO (umbral = 10 MB) ===")
println("(Spark decide solo porque 'categorias' cabe en el umbral)")
println("─" * 60)
autobroadcast.explain()

println("\n✅ Resultado idéntico (mismas filas, misma lógica):")
println(s"  Filas sin broadcast:  ${sinBroadcast.count()}")
println(s"  Filas con broadcast:  ${conBroadcast.count()}")
println(s"  Filas auto broadcast: ${autobroadcast.count()}")
println("  Los tres resultados son iguales — solo cambia la estrategia interna.")

Umbral autoBroadcastJoinThreshold: 10485760 bytes (10 MB)

=== Plan con broadcast AUTOMÁTICO (umbral = 10 MB) ===
(Spark decide solo porque 'categorias' cabe en el umbral)
────────────────────────────────────────────────────────────
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- BroadcastHashJoin [categoriaId#287], [id#302], Inner, BuildRight, false
   :- LocalTableScan [pedidoId#285, clienteId#286, categoriaId#287, importe#288]
   +- BroadcastExchange HashedRelationBroadcastMode(List(input[0, string, true]),false), [plan_id=1368]
      +- LocalTableScan [id#302, nombreCategoria#303, segmento#304]



✅ Resultado idéntico (mismas filas, misma lógica):
  Filas sin broadcast:  12
  Filas con broadcast:  12
  Filas auto broadcast: 12
  Los tres resultados son iguales — solo cambia la estrategia interna.


import org.apache.spark.sql.functions.broadcast
umbralActivo: String = "10485760"
autobroadcast: org.apache.spark.sql.package.DataFrame = [pedidoId: string, clienteId: int ... 5 more fields]

In [18]:
import $ivy.`org.apache.spark::spark-sql:4.1.1`

import org.apache.spark.sql.SparkSession
import org.apache.spark.sql.functions._

val spark = SparkSession.builder()
  .appName("Dia18-Sesion1-Joins")
  .master("local[*]")
  .config("spark.sql.shuffle.partitions", "4")
  .config("spark.sql.crossJoin.enabled",  "true")
  .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
import spark.implicits._

println(s"Spark ${spark.version} — Scala ${scala.util.Properties.versionString} ✅")

Spark 4.1.1 — Scala version 2.13.18 ✅


import $ivy.$
import org.apache.spark.sql.SparkSession
import org.apache.spark.sql.functions._
spark: SparkSession = org.apache.spark.sql.classic.SparkSession@55981713
import spark.implicits._

In [19]:
// === CLIENTES ===
val clientes = Seq(
  (1, "Ana García",    "Madrid"),
  (2, "Borja Ruiz",    "Barcelona"),
  (3, "Carmen López",  "Sevilla"),
  (4, "David Mora",    "Valencia"),
  (5, "Elena Pardo",   "Bilbao")
).toDF("clienteId", "nombre", "ciudad")

// === PEDIDOS ===
// clienteId 6 no existe en el maestro de clientes → dato huérfano
val pedidos = Seq(
  ("P001", 1, "2024-01-10", 250.0, "R01"),
  ("P002", 1, "2024-01-15", 180.0, "R02"),
  ("P003", 2, "2024-01-20", 420.0, "R01"),
  ("P004", 3, "2024-02-01",  95.0, "R03"),
  ("P005", 3, "2024-02-14",  60.0, "R02"),
  ("P006", 6, "2024-02-20", 310.0, "R03")  // ← cliente 6 no existe
).toDF("pedidoId", "clienteId", "fecha", "importe", "repartidorId")

// === REPARTIDORES ===
val repartidores = Seq(
  ("R01", "Miguel Sanz",   "Zona Norte"),
  ("R02", "Laura Vega",    "Zona Sur"),
  ("R03", "Pablo Fuentes", "Zona Este")
).toDF("repartidorId", "nombreRep", "zona")

println("DataFrames creados:")
println(s"  clientes:     ${clientes.count()} filas")
println(s"  pedidos:      ${pedidos.count()} filas")
println(s"  repartidores: ${repartidores.count()} filas")

DataFrames creados:
  clientes:     5 filas
  pedidos:      6 filas
  repartidores: 3 filas


clientes: org.apache.spark.sql.package.DataFrame = [clienteId: int, nombre: string ... 1 more field]
pedidos: org.apache.spark.sql.package.DataFrame = [pedidoId: string, clienteId: int ... 3 more fields]
repartidores: org.apache.spark.sql.package.DataFrame = [repartidorId: string, nombreRep: string ... 1 more field]

In [20]:
// Buena práctica: renombrar la columna conflictiva ANTES del join
val pedidosRen = pedidos.withColumnRenamed("clienteId", "pedido_clienteId")

val resultado_inner = clientes.join(
  pedidosRen,
  col("clienteId") === col("pedido_clienteId"),
  "inner"
).select(
  col("pedidoId"),
  col("nombre").alias("cliente"),
  col("ciudad"),
  col("fecha"),
  col("importe")
).orderBy("pedidoId")

println("=== INNER JOIN: pedidos con cliente conocido ===")
resultado_inner.show()
println(s"Total filas: ${resultado_inner.count()}")
// P006 desaparece porque clienteId=6 no existe → esperamos 5 filas

=== INNER JOIN: pedidos con cliente conocido ===
+--------+------------+---------+----------+-------+
|pedidoId|     cliente|   ciudad|     fecha|importe|
+--------+------------+---------+----------+-------+
|    P001|  Ana García|   Madrid|2024-01-10|  250.0|
|    P002|  Ana García|   Madrid|2024-01-15|  180.0|
|    P003|  Borja Ruiz|Barcelona|2024-01-20|  420.0|
|    P004|Carmen López|  Sevilla|2024-02-01|   95.0|
|    P005|Carmen López|  Sevilla|2024-02-14|   60.0|
+--------+------------+---------+----------+-------+

Total filas: 5


pedidosRen: org.apache.spark.sql.package.DataFrame = [pedidoId: string, pedido_clienteId: int ... 3 more fields]
resultado_inner: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [pedidoId: string, cliente: string ... 3 more fields]

In [21]:
val resultado_left = clientes.join(
  pedidosRen,
  col("clienteId") === col("pedido_clienteId"),
  "left"
).select(
  col("clienteId"),
  col("nombre"),
  col("pedidoId"),
  col("importe")
).orderBy("clienteId", "pedidoId")

println("=== LEFT JOIN: todos los clientes (con o sin pedidos) ===")
resultado_left.show()
// David Mora y Elena Pardo aparecen con null en pedidoId e importe

=== LEFT JOIN: todos los clientes (con o sin pedidos) ===
+---------+------------+--------+-------+
|clienteId|      nombre|pedidoId|importe|
+---------+------------+--------+-------+
|        1|  Ana García|    P001|  250.0|
|        1|  Ana García|    P002|  180.0|
|        2|  Borja Ruiz|    P003|  420.0|
|        3|Carmen López|    P004|   95.0|
|        3|Carmen López|    P005|   60.0|
|        4|  David Mora|    NULL|   NULL|
|        5| Elena Pardo|    NULL|   NULL|
+---------+------------+--------+-------+



resultado_left: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [clienteId: int, nombre: string ... 2 more fields]

In [22]:
val resultado_left = clientes.join(
  pedidosRen,
  col("clienteId") === col("pedido_clienteId"),
  "left"
).select(
  col("clienteId"),
  col("nombre"),
  col("pedidoId"),
  col("importe")
).orderBy("clienteId", "pedidoId")

println("=== LEFT JOIN: todos los clientes (con o sin pedidos) ===")
resultado_left.show()
// David Mora y Elena Pardo aparecen con null en pedidoId e importe

=== LEFT JOIN: todos los clientes (con o sin pedidos) ===
+---------+------------+--------+-------+
|clienteId|      nombre|pedidoId|importe|
+---------+------------+--------+-------+
|        1|  Ana García|    P001|  250.0|
|        1|  Ana García|    P002|  180.0|
|        2|  Borja Ruiz|    P003|  420.0|
|        3|Carmen López|    P004|   95.0|
|        3|Carmen López|    P005|   60.0|
|        4|  David Mora|    NULL|   NULL|
|        5| Elena Pardo|    NULL|   NULL|
+---------+------------+--------+-------+



resultado_left: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [clienteId: int, nombre: string ... 2 more fields]

In [23]:
// LEFT SEMI: clientes que SÍ tienen al menos un pedido
val clientesConPedidos = clientes.join(
  pedidosRen,
  col("clienteId") === col("pedido_clienteId"),
  "semi"
)

println("=== LEFT SEMI: clientes que han hecho algún pedido ===")
clientesConPedidos.show()
// Solo columnas de clientes; David Mora y Elena Pardo no aparecen

// LEFT ANTI: clientes que NUNCA han pedido
val clientesSinPedidos = clientes.join(
  pedidosRen,
  col("clienteId") === col("pedido_clienteId"),
  "anti"
)

println("=== LEFT ANTI: clientes sin ningún pedido ===")
clientesSinPedidos.show()
// Solo David Mora y Elena Pardo

=== LEFT SEMI: clientes que han hecho algún pedido ===
+---------+------------+---------+
|clienteId|      nombre|   ciudad|
+---------+------------+---------+
|        1|  Ana García|   Madrid|
|        2|  Borja Ruiz|Barcelona|
|        3|Carmen López|  Sevilla|
+---------+------------+---------+

=== LEFT ANTI: clientes sin ningún pedido ===
+---------+-----------+--------+
|clienteId|     nombre|  ciudad|
+---------+-----------+--------+
|        4| David Mora|Valencia|
|        5|Elena Pardo|  Bilbao|
+---------+-----------+--------+



clientesConPedidos: org.apache.spark.sql.package.DataFrame = [clienteId: int, nombre: string ... 1 more field]
clientesSinPedidos: org.apache.spark.sql.package.DataFrame = [clienteId: int, nombre: string ... 1 more field]

In [24]:
// Renombramos todas las columnas que pueden colisionar
val pedidos3 = pedidos
  .withColumnRenamed("clienteId",    "pedido_clienteId")
  .withColumnRenamed("repartidorId", "pedido_repartidorId")

val resultado_triple = pedidos3
  .join(clientes,     col("pedido_clienteId")    === col("clienteId"),    "inner")
  .join(repartidores, col("pedido_repartidorId") === col("repartidorId"), "inner")
  .select(
    col("pedidoId"),
    col("nombre").alias("cliente"),
    col("ciudad"),
    col("fecha"),
    col("importe"),
    col("nombreRep").alias("repartidor"),
    col("zona")
  )
  .orderBy("pedidoId")

println("=== JOIN TRIPLE: pedidos + clientes + repartidores ===")
resultado_triple.show(truncate = false)

=== JOIN TRIPLE: pedidos + clientes + repartidores ===
+--------+------------+---------+----------+-------+-------------+----------+
|pedidoId|cliente     |ciudad   |fecha     |importe|repartidor   |zona      |
+--------+------------+---------+----------+-------+-------------+----------+
|P001    |Ana García  |Madrid   |2024-01-10|250.0  |Miguel Sanz  |Zona Norte|
|P002    |Ana García  |Madrid   |2024-01-15|180.0  |Laura Vega   |Zona Sur  |
|P003    |Borja Ruiz  |Barcelona|2024-01-20|420.0  |Miguel Sanz  |Zona Norte|
|P004    |Carmen López|Sevilla  |2024-02-01|95.0   |Pablo Fuentes|Zona Este |
|P005    |Carmen López|Sevilla  |2024-02-14|60.0   |Laura Vega   |Zona Sur  |
+--------+------------+---------+----------+-------+-------------+----------+



pedidos3: org.apache.spark.sql.package.DataFrame = [pedidoId: string, pedido_clienteId: int ... 3 more fields]
resultado_triple: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [pedidoId: string, cliente: string ... 5 more fields]

In [25]:
val pedidos4 = pedidos
  .withColumnRenamed("repartidorId", "pedido_repartidorId")

// Sin broadcast
val sinBroadcast = pedidos4.join(
  repartidores,
  col("pedido_repartidorId") === col("repartidorId"),
  "inner"
)

println("=== Plan SIN broadcast ===")
sinBroadcast.explain()

// Con broadcast forzado (repartidores es pequeño)
val conBroadcast = pedidos4.join(
  broadcast(repartidores),
  col("pedido_repartidorId") === col("repartidorId"),
  "inner"
)

println("\n=== Plan CON broadcast ===")
conBroadcast.explain()

// Los resultados deben ser idénticos
println(s"\nFilas sin broadcast: ${sinBroadcast.count()}")
println(s"Filas con broadcast: ${conBroadcast.count()}")

=== Plan SIN broadcast ===
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- BroadcastHashJoin [pedido_repartidorId#588], [repartidorId#463], Inner, BuildRight, false
   :- LocalTableScan [pedidoId#445, clienteId#446, fecha#447, importe#448, pedido_repartidorId#588]
   +- BroadcastExchange HashedRelationBroadcastMode(List(input[0, string, true]),false), [plan_id=2160]
      +- LocalTableScan [repartidorId#463, nombreRep#464, zona#465]



=== Plan CON broadcast ===
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- BroadcastHashJoin [pedido_repartidorId#588], [repartidorId#463], Inner, BuildRight, false
   :- LocalTableScan [pedidoId#445, clienteId#446, fecha#447, importe#448, pedido_repartidorId#588]
   +- BroadcastExchange HashedRelationBroadcastMode(List(input[0, string, true]),false), [plan_id=2174]
      +- LocalTableScan [repartidorId#463, nombreRep#464, zona#465]



Filas sin broadcast: 6
Filas con broadcast: 6


pedidos4: org.apache.spark.sql.package.DataFrame = [pedidoId: string, clienteId: int ... 3 more fields]
sinBroadcast: org.apache.spark.sql.package.DataFrame = [pedidoId: string, clienteId: int ... 6 more fields]
conBroadcast: org.apache.spark.sql.package.DataFrame = [pedidoId: string, clienteId: int ... 6 more fields]

In [26]:
val ciudadesEnero = Seq(
  ("Madrid",    "Norte"),
  ("Barcelona", "Este"),
  ("Sevilla",   "Sur")
).toDF("ciudad", "zona")

val ciudadesFebrero = Seq(
  ("Madrid",   "Norte"),
  ("Valencia", "Este"),
  ("Bilbao",   "Norte")
).toDF("ciudad", "zona")

// UNION con duplicados (Madrid aparece dos veces)
val todasConDup = ciudadesEnero.union(ciudadesFebrero)
println(s"=== union (con duplicados): ${todasConDup.count()} filas ===")
todasConDup.show()

// UNION sin duplicados
val todasSinDup = ciudadesEnero.union(ciudadesFebrero).distinct()
println(s"=== union.distinct (sin duplicados): ${todasSinDup.count()} filas ===")
todasSinDup.show()

// EXCEPT: ciudades de enero que NO están en febrero
val soloEnero = ciudadesEnero.except(ciudadesFebrero)
println("=== except: ciudades solo cubiertas en enero ===")
soloEnero.show()
// Esperamos: Barcelona y Sevilla

=== union (con duplicados): 6 filas ===
+---------+-----+
|   ciudad| zona|
+---------+-----+
|   Madrid|Norte|
|Barcelona| Este|
|  Sevilla|  Sur|
|   Madrid|Norte|
| Valencia| Este|
|   Bilbao|Norte|
+---------+-----+

=== union.distinct (sin duplicados): 5 filas ===
+---------+-----+
|   ciudad| zona|
+---------+-----+
|   Madrid|Norte|
|Barcelona| Este|
|  Sevilla|  Sur|
| Valencia| Este|
|   Bilbao|Norte|
+---------+-----+

=== except: ciudades solo cubiertas en enero ===
+---------+----+
|   ciudad|zona|
+---------+----+
|Barcelona|Este|
|  Sevilla| Sur|
+---------+----+



ciudadesEnero: org.apache.spark.sql.package.DataFrame = [ciudad: string, zona: string]
ciudadesFebrero: org.apache.spark.sql.package.DataFrame = [ciudad: string, zona: string]
todasConDup: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [ciudad: string, zona: string]
todasSinDup: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [ciudad: string, zona: string]
soloEnero: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [ciudad: string, zona: string]

In [27]:
println("=" * 52)
println("RESUMEN — Día 15 Sesión 2 | LogiData S.A.")
println("=" * 52)

val checks = Seq(
  ("INNER JOIN — pedidos con cliente válido",     resultado_inner.count()    == 5),
  ("LEFT ANTI  — clientes sin pedidos",           clientesSinPedidos.count() == 2),
  ("JOIN TRIPLE — pedidos+clientes+repartidores", resultado_triple.count()   == 5),
  ("UNION.DISTINCT — sin duplicados",             todasSinDup.count()        == 5),
  ("EXCEPT — solo ciudades de enero",             soloEnero.count()          == 2)
)

checks.foreach { case (desc, ok) =>
  println(s"${if (ok) "✅ CORRECTO" else "❌ REVISAR"} — $desc")
}

RESUMEN — Día 15 Sesión 2 | LogiData S.A.
✅ CORRECTO — INNER JOIN — pedidos con cliente válido
✅ CORRECTO — LEFT ANTI  — clientes sin pedidos
✅ CORRECTO — JOIN TRIPLE — pedidos+clientes+repartidores
✅ CORRECTO — UNION.DISTINCT — sin duplicados
✅ CORRECTO — EXCEPT — solo ciudades de enero


checks: Seq[(String, Boolean)] = List(
  ("INNER JOIN — pedidos con cliente válido", true),
  ("LEFT ANTI  — clientes sin pedidos", true),
  ("JOIN TRIPLE — pedidos+clientes+repartidores", true),
  ("UNION.DISTINCT — sin duplicados", true),
  ("EXCEPT — solo ciudades de enero", true)
)

In [ ]:
🏥 Caso de Estudio - MediRed S.A.. Análisis de la Red Nacional de Farmacias

In [28]:
import $ivy.`org.apache.spark::spark-sql:4.1.1`

import org.apache.spark.sql.SparkSession
import org.apache.spark.sql.functions._

val spark = SparkSession.builder()
  .appName("CasoEstudio-MediRed")
  .master("local[*]")
  .config("spark.sql.shuffle.partitions", "4")
  .config("spark.sql.crossJoin.enabled",  "true")
  .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
import spark.implicits._

println(s"Spark ${spark.version} — Scala ${scala.util.Properties.versionString} ✅")

Spark 4.1.1 — Scala version 2.13.18 ✅


import $ivy.$
import org.apache.spark.sql.SparkSession
import org.apache.spark.sql.functions._
spark: SparkSession = org.apache.spark.sql.classic.SparkSession@55981713
import spark.implicits._

In [29]:
// === FARMACIAS ===
val farmacias = Seq(
  (1,  "Farmacia Central",     "Madrid",    "Madrid",    "Laura Vega"),
  (2,  "Farmacia del Sol",     "Madrid",    "Madrid",    "Carlos Ruiz"),
  (3,  "Farmacia Diagonal",    "Barcelona", "Cataluña",  "Ana Puig"),
  (4,  "Farmacia Rambla",      "Barcelona", "Cataluña",  "Marta Font"),
  (5,  "Farmacia Norte",       "Bilbao",    "País Vasco", "Jon Etxea"),
  (6,  "Farmacia Ría",         "Bilbao",    "País Vasco", "Amaia Goñi"),
  (7,  "Farmacia Gran Vía",    "Sevilla",   "Andalucía", "Pedro Mora"),
  (8,  "Farmacia Triana",      "Sevilla",   "Andalucía", "Rosa Leal")
).toDF("farmaciaId", "nombre", "ciudad", "comunidad", "titular")

// === VENTAS ===
// farmaciaId 99 no existe en el maestro → venta huérfana del proveedor externo
val ventas = Seq(
  ("V001",  1, "P-AMOX", 3, 18.50, "2024-01-05"),
  ("V002",  1, "P-IBUP",10, 42.00, "2024-01-07"),
  ("V003",  2, "P-VITA", 5, 25.00, "2024-01-08"),
  ("V004",  3, "P-AMOX", 2, 12.50, "2024-01-10"),
  ("V005",  3, "P-PARA", 8, 16.00, "2024-01-12"),
  ("V006",  4, "P-IBUP", 6, 24.00, "2024-01-15"),
  ("V007",  5, "P-VITA",12, 60.00, "2024-01-18"),
  ("V008",  6, "P-PARA", 4,  8.00, "2024-01-20"),
  ("V009",  7, "P-AMOX", 7, 43.75, "2024-02-01"),
  ("V010",  7, "P-IBUP", 3, 12.00, "2024-02-03"),
  ("V011",  8, "P-VITA", 9, 45.00, "2024-02-05"),
  ("V012", 99, "P-PARA", 2,  4.00, "2024-02-10")  // ← farmaciaId 99 no existe
).toDF("ventaId", "farmaciaId", "productoId", "cantidad", "importe", "fecha")

// === PRODUCTOS (tabla pequeña: 4 registros) ===
val productos = Seq(
  ("P-AMOX", "Amoxicilina 500mg",  "Antibiótico",  true),
  ("P-IBUP", "Ibuprofeno 600mg",   "Analgésico",   false),
  ("P-VITA", "Vitamina D 1000 UI", "Vitamina",     false),
  ("P-PARA", "Paracetamol 1g",     "Analgésico",   false)
).toDF("productoId", "nombreProducto", "categoria", "requiereReceta")

// === INSPECCIONES Q1 (enero-marzo) ===
val inspeccionesQ1 = Seq(
  (1, "2024-01-15", "Apto"),
  (2, "2024-01-22", "Apto"),
  (3, "2024-02-05", "No Apto"),
  (5, "2024-02-18", "Apto"),
  (7, "2024-03-10", "Apto")
).toDF("farmaciaId", "fecha", "resultado")

// === INSPECCIONES Q2 (abril-junio) ===
val inspeccionesQ2 = Seq(
  (3, "2024-04-08", "Apto"),    // corrigió el No Apto del Q1
  (4, "2024-04-20", "Apto"),
  (6, "2024-05-12", "No Apto"),
  (7, "2024-05-25", "Apto"),
  (8, "2024-06-03", "Apto")
).toDF("farmaciaId", "fecha", "resultado")

println("Datos cargados:")
println(s"  farmacias:      ${farmacias.count()} registros")
println(s"  ventas:         ${ventas.count()} registros")
println(s"  productos:      ${productos.count()} registros")
println(s"  inspeccionesQ1: ${inspeccionesQ1.count()} registros")
println(s"  inspeccionesQ2: ${inspeccionesQ2.count()} registros")

Datos cargados:
  farmacias:      8 registros
  ventas:         12 registros
  productos:      4 registros
  inspeccionesQ1: 5 registros
  inspeccionesQ2: 5 registros


farmacias: org.apache.spark.sql.package.DataFrame = [farmaciaId: int, nombre: string ... 3 more fields]
ventas: org.apache.spark.sql.package.DataFrame = [ventaId: string, farmaciaId: int ... 4 more fields]
productos: org.apache.spark.sql.package.DataFrame = [productoId: string, nombreProducto: string ... 2 more fields]
inspeccionesQ1: org.apache.spark.sql.package.DataFrame = [farmaciaId: int, fecha: string ... 1 more field]
inspeccionesQ2: org.apache.spark.sql.package.DataFrame = [farmaciaId: int, fecha: string ... 1 more field]

In [30]:
import org.apache.spark.sql.functions.{broadcast, col}


val productosRenombrado = productos.withColumnRenamed("productoId", "p_id")

val informe = ventas.join(

    broadcast(productosRenombrado), 
    ventas("productoId") === productosRenombrado("p_id"), 
    "inner"
)

val resultado = informe.select(
    "ventaId", 
    "nombreProducto", 
    "categoria", 
    "requiereReceta", 
    "cantidad", 
    "importe"
).orderBy(col("importe").desc)

resultado.show()


+-------+------------------+-----------+--------------+--------+-------+
|ventaId|    nombreProducto|  categoria|requiereReceta|cantidad|importe|
+-------+------------------+-----------+--------------+--------+-------+
|   V007|Vitamina D 1000 UI|   Vitamina|         false|      12|   60.0|
|   V011|Vitamina D 1000 UI|   Vitamina|         false|       9|   45.0|
|   V009| Amoxicilina 500mg|Antibiótico|          true|       7|  43.75|
|   V002|  Ibuprofeno 600mg| Analgésico|         false|      10|   42.0|
|   V003|Vitamina D 1000 UI|   Vitamina|         false|       5|   25.0|
|   V006|  Ibuprofeno 600mg| Analgésico|         false|       6|   24.0|
|   V001| Amoxicilina 500mg|Antibiótico|          true|       3|   18.5|
|   V005|    Paracetamol 1g| Analgésico|         false|       8|   16.0|
|   V004| Amoxicilina 500mg|Antibiótico|          true|       2|   12.5|
|   V010|  Ibuprofeno 600mg| Analgésico|         false|       3|   12.0|
|   V008|    Paracetamol 1g| Analgésico|         fa

import org.apache.spark.sql.functions.{broadcast, col}
productosRenombrado: org.apache.spark.sql.package.DataFrame = [p_id: string, nombreProducto: string ... 2 more fields]
informe: org.apache.spark.sql.package.DataFrame = [ventaId: string, farmaciaId: int ... 8 more fields]
resultado: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [ventaId: string, nombreProducto: string ... 4 more fields]

In [ ]:
Es inner join ya que mantiene las filas en ambas tablas
Si porque cuando es pqequeña es tan bien dicha optimización de broadcast.

In [ ]:
import org.apache.spark.sql.functions.col

val farmaciasmaster = farmacias.withColumnRenamed("farmaciaId", "id_master")


val ventasSinMaestro = ventas.join(
    ventasRenombrado,
    farmacias("farmaciaId") === ventasRenombrado("f_id"),
    "left_anti"
)


val resultado = farmaciasSinVentas.select("farmaciaId", "nombre", "ciudad")

resultado.show()


+----------+------+------+
|farmaciaId|nombre|ciudad|
+----------+------+------+
+----------+------+------+



import org.apache.spark.sql.functions.col
ventasRenombrado: org.apache.spark.sql.package.DataFrame = [ventaId: string, f_id: int ... 4 more fields]
farmaciasSinVentas: org.apache.spark.sql.package.DataFrame = [farmaciaId: int, nombre: string ... 3 more fields]
resultado: org.apache.spark.sql.package.DataFrame = [farmaciaId: int, nombre: string ... 1 more field]

In [33]:
import org.apache.spark.sql.functions.col


val farmaciasMaster = farmacias.withColumnRenamed("farmaciaId", "id_master")


val ventasSinMaestro = ventas.join(
    farmaciasMaster, 
    ventas("farmaciaId") === farmaciasMaster("id_master"), 
    "left_anti"
)


val resultadoAuditoria = ventasSinMaestro.select(
    "ventaId", 
    "farmaciaId", 
    "productoId", 
    "importe"
)

resultadoAuditoria.show()


+-------+----------+----------+-------+
|ventaId|farmaciaId|productoId|importe|
+-------+----------+----------+-------+
|   V012|        99|    P-PARA|    4.0|
+-------+----------+----------+-------+



import org.apache.spark.sql.functions.col
farmaciasMaster: org.apache.spark.sql.package.DataFrame = [id_master: int, nombre: string ... 3 more fields]
ventasSinMaestro: org.apache.spark.sql.package.DataFrame = [ventaId: string, farmaciaId: int ... 4 more fields]
resultadoAuditoria: org.apache.spark.sql.package.DataFrame = [ventaId: string, farmaciaId: int ... 2 more fields]

In [ ]:
EL dataframe que debería ir a la izquierda es el de las ventas.

In [34]:
import org.apache.spark.sql.functions.col

val v = ventas.withColumnRenamed("farmaciaId", "v_fId")
              .withColumnRenamed("productoId", "v_pId")

val f = farmacias.withColumnRenamed("farmaciaId", "f_id")
val p = productos.withColumnRenamed("productoId", "p_id")

val informeEjecutivo = v
  .join(f, v("v_fId") === f("f_id"), "inner")
  .join(p, v("v_pId") === p("p_id"), "inner")


val resultadoFinal = informeEjecutivo.select(
    col("ventaId"),
    col("nombre").as("nombre_farmacia"), 
    col("comunidad"),
    col("nombreProducto"),
    col("categoria"),
    col("cantidad"),
    col("importe"),
    col("fecha")
).orderBy(col("comunidad").asc, col("importe").desc)

resultadoFinal.show()


+-------+-----------------+----------+------------------+-----------+--------+-------+----------+
|ventaId|  nombre_farmacia| comunidad|    nombreProducto|  categoria|cantidad|importe|     fecha|
+-------+-----------------+----------+------------------+-----------+--------+-------+----------+
|   V011|  Farmacia Triana| Andalucía|Vitamina D 1000 UI|   Vitamina|       9|   45.0|2024-02-05|
|   V009|Farmacia Gran Vía| Andalucía| Amoxicilina 500mg|Antibiótico|       7|  43.75|2024-02-01|
|   V010|Farmacia Gran Vía| Andalucía|  Ibuprofeno 600mg| Analgésico|       3|   12.0|2024-02-03|
|   V006|  Farmacia Rambla|  Cataluña|  Ibuprofeno 600mg| Analgésico|       6|   24.0|2024-01-15|
|   V005|Farmacia Diagonal|  Cataluña|    Paracetamol 1g| Analgésico|       8|   16.0|2024-01-12|
|   V004|Farmacia Diagonal|  Cataluña| Amoxicilina 500mg|Antibiótico|       2|   12.5|2024-01-10|
|   V002| Farmacia Central|    Madrid|  Ibuprofeno 600mg| Analgésico|      10|   42.0|2024-01-07|
|   V003| Farmacia d

import org.apache.spark.sql.functions.col
v: org.apache.spark.sql.package.DataFrame = [ventaId: string, v_fId: int ... 4 more fields]
f: org.apache.spark.sql.package.DataFrame = [f_id: int, nombre: string ... 3 more fields]
p: org.apache.spark.sql.package.DataFrame = [p_id: string, nombreProducto: string ... 2 more fields]
informeEjecutivo: org.apache.spark.sql.package.DataFrame = [ventaId: string, v_fId: int ... 13 more fields]
resultadoFinal: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [ventaId: string, nombre_farmacia: string ... 6 more fields]

In [ ]:

val IDsConInspeccionCompleta = inspeccionesQ1.select("farmaciaId")
  .intersect(inspeccionesQ2.select("farmaciaId"))


val resultadoSanidad = IDsConInspeccionCompleta.join(
    farmacias, 
    "farmaciaId"
)

resultadoSanidad.select("farmaciaId", "nombre", "ciudad").show()


+----------+-----------------+---------+
|farmaciaId|           nombre|   ciudad|
+----------+-----------------+---------+
|         3|Farmacia Diagonal|Barcelona|
|         7|Farmacia Gran Vía|  Sevilla|
+----------+-----------------+---------+



IDsConInspeccionCompleta: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [farmaciaId: int]
resultadoSanidad: org.apache.spark.sql.package.DataFrame = [farmaciaId: int, nombre: string ... 3 more fields]

In [36]:
import org.apache.spark.sql.functions.col

val todasLasInspecciones = inspeccionesQ1.union(inspeccionesQ2)

val historialUnico =todasLasInspecciones.distinct()


val resultadoFinal = historialUnico
.orderBy(col("fecha").asc)


println(s"Total con duplicados : ${todasLasInspecciones.count()}")
println(s"Total único : ${historialUnico.count()}")


resultadoFinal.show()


Total con duplicados : 10
Total único : 10
+----------+----------+---------+
|farmaciaId|     fecha|resultado|
+----------+----------+---------+
|         1|2024-01-15|     Apto|
|         2|2024-01-22|     Apto|
|         3|2024-02-05|  No Apto|
|         5|2024-02-18|     Apto|
|         7|2024-03-10|     Apto|
|         3|2024-04-08|     Apto|
|         4|2024-04-20|     Apto|
|         6|2024-05-12|  No Apto|
|         7|2024-05-25|     Apto|
|         8|2024-06-03|     Apto|
+----------+----------+---------+



import org.apache.spark.sql.functions.col
todasLasInspecciones: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [farmaciaId: int, fecha: string ... 1 more field]
historialUnico: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [farmaciaId: int, fecha: string ... 1 more field]
resultadoFinal: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [farmaciaId: int, fecha: string ... 1 more field]

In [ ]:

val idsSoloEnQ1 = inspeccionesQ1.select("farmaciaId").
  except(inspeccionesQ2.select("farmaciaId"))

val farmaciasPendientes = idsSoloEnQ1.join(
    farmacias,
    "farmaciaId"
)

val resultadoFinal = farmaciasPendientes.select("farmaciaId", "nombre", "titular")

resultadoFinal.show() 


+----------+----------------+-----------+
|farmaciaId|          nombre|    titular|
+----------+----------------+-----------+
|         1|Farmacia Central| Laura Vega|
|         2|Farmacia del Sol|Carlos Ruiz|
|         5|  Farmacia Norte|  Jon Etxea|
+----------+----------------+-----------+



idsSoloEnQ1: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [farmaciaId: int]
farmaciasPendientes: org.apache.spark.sql.package.DataFrame = [farmaciaId: int, nombre: string ... 3 more fields]
resultadoFinal: org.apache.spark.sql.package.DataFrame = [farmaciaId: int, nombre: string ... 1 more field]

In [ ]:
import org.apache.spark.sql.functions.lit


val farmaciasDS = Seq(
  (1, "Farmacia Central"), (2, "Farmacia Norte"), (3, "Farmacia Sur"), (4, "Farmacia Este"),
  (5, "Farmacia Oeste"), (6, "Farmacia Salud"), (7, "Farmacia Vida"), (8, "Farmacia Bienestar")
).toDF("farmaciaId", "nombre")


val categoriasDS = Seq("Medicamentos", "Higiene", "Belleza").toDF("categoria")


val planAuditoriaDF = farmaciasDS
  .join(categoriasDS, lit(true), "cross")
  .orderBy("farmaciaId")


planAuditoriaDF.show(24)


+----------+------------------+------------+
|farmaciaId|            nombre|   categoria|
+----------+------------------+------------+
|         1|  Farmacia Central|Medicamentos|
|         1|  Farmacia Central|     Higiene|
|         1|  Farmacia Central|     Belleza|
|         2|    Farmacia Norte|Medicamentos|
|         2|    Farmacia Norte|     Higiene|
|         2|    Farmacia Norte|     Belleza|
|         3|      Farmacia Sur|Medicamentos|
|         3|      Farmacia Sur|     Higiene|
|         3|      Farmacia Sur|     Belleza|
|         4|     Farmacia Este|Medicamentos|
|         4|     Farmacia Este|     Higiene|
|         4|     Farmacia Este|     Belleza|
|         5|    Farmacia Oeste|Medicamentos|
|         5|    Farmacia Oeste|     Higiene|
|         5|    Farmacia Oeste|     Belleza|
|         6|    Farmacia Salud|Medicamentos|
|         6|    Farmacia Salud|     Higiene|
|         6|    Farmacia Salud|     Belleza|
|         7|     Farmacia Vida|Medicamentos|
|         

import org.apache.spark.sql.functions.lit
farmaciasDS: org.apache.spark.sql.package.DataFrame = [farmaciaId: int, nombre: string]
categoriasDS: org.apache.spark.sql.package.DataFrame = [categoria: string]
planAuditoriaDF: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [farmaciaId: int, nombre: string ... 1 more field]

In [48]:
import org.apache.spark.sql.functions.broadcast

val df_productos = Seq(
  (1, "Paracetamol", "Medicamentos"),
  (2, "Jabón", "Higiene"),
  (3, "Crema Facial", "Belleza")
).toDF("productoId", "nombreProducto", "categoria")

val df_ventas = Seq(
  (101, 1, 2, "2023-10-01"),
  (102, 2, 1, "2023-10-01"),
  (103, 1, 5, "2023-10-02")
).toDF("ventaId", "productoId", "cantidad", "fecha")

println("--- PLAN SIN BROADCAST ---")
val joinSin = df_ventas.join(df_productos, "productoId")
joinSin.explain()


println("--- PLAN CON BROADCAST ---")
val joinCon = df_ventas.join(broadcast(df_productos), "productoId")
joinCon.explain()


--- PLAN SIN BROADCAST ---
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [productoId#1041, ventaId#1040, cantidad#1042, fecha#1043, nombreProducto#1021, categoria#1022]
   +- BroadcastHashJoin [productoId#1041], [productoId#1020], Inner, BuildLeft, false
      :- BroadcastExchange HashedRelationBroadcastMode(List(cast(input[1, int, false] as bigint)),false), [plan_id=4311]
      :  +- LocalTableScan [ventaId#1040, productoId#1041, cantidad#1042, fecha#1043]
      +- LocalTableScan [productoId#1020, nombreProducto#1021, categoria#1022]


--- PLAN CON BROADCAST ---
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [productoId#1041, ventaId#1040, cantidad#1042, fecha#1043, nombreProducto#1021, categoria#1022]
   +- BroadcastHashJoin [productoId#1041], [productoId#1020], Inner, BuildRight, false
      :- LocalTableScan [ventaId#1040, productoId#1041, cantidad#1042, fecha#1043]
      +- BroadcastExchange HashedRelationBroadcastMode(List(cast(input[0, in

import org.apache.spark.sql.functions.broadcast
df_productos: org.apache.spark.sql.package.DataFrame = [productoId: int, nombreProducto: string ... 1 more field]
df_ventas: org.apache.spark.sql.package.DataFrame = [ventaId: int, productoId: int ... 2 more fields]
joinSin: org.apache.spark.sql.package.DataFrame = [productoId: int, ventaId: int ... 4 more fields]
joinCon: org.apache.spark.sql.package.DataFrame = [productoId: int, ventaId: int ... 4 more fields]

1. ¿Qué tipo de join aparece en el plan sin broadcast?Generalmente aparece un SortMergeJoin.

2. ¿Qué tipo de join aparece en el plan con broadcast?Aparece un BroadcastHashJoin.
3. ¿Por qué tiene sentido aplicar broadcast sobre productos y no sobre ventas?Porque en productos es pequeña y estática y en ventas crece continuamente
4. ¿Qué columna del DataFrame ventas es la más adecuada para hacer el join con productos?la columna productoId

In [ ]:
import org.apache.spark.sql.functions.lit

val prod = Seq((1, "P", "Cat1"), (2, "J", "Cat2"), (3, "C", "Cat3")).toDF("productoId", "nombre", "categoria")
val farm = (1 to 8).map(i => (i, s"Farmacia $i")).toDF("farmaciaId", "nombre")
val vQ1 = Seq((1, 1), (2, 2)).toDF("farmaciaId", "productoId")
val vQ2 = Seq((1, 1), (3, 2)).toDF("farmaciaId", "productoId")
val vent = Seq((99, 99)).toDF("farmaciaId", "productoId") // Para simular huérfanas


val ventasHuerfanas = vent.join(prod, Seq("productoId"), "left_anti")
val farmaciasAmbosTrims = vQ1.join(vQ2, "farmaciaId", "inner").select("farmaciaId").distinct()
val soloQ1 = vQ1.join(vQ2, Seq("farmaciaId"), "left_anti").select("farmaciaId").distinct()
val matrizAuditoria = farm.select("farmaciaId", "nombre")
  .join(prod.select("categoria").distinct(), lit(true), "cross")


println("=" * 55)
println("VERIFICACIÓN FINAL - Caso de Estudio MediRed S.A.")
println("=" * 55)

val comprobaciones = Seq(
  ("Misión 3 - Ventas huérfanas detectadas",    ventasHuerfanas.count() > 0),
  ("Misión 5 - Farmacias con doble inspección", farmaciasAmbosTrims.count() > 0),
  ("Misión 7 - Farmacias solo en Q1",           soloQ1.count() > 0),
  ("Misión 8 - Matriz auditoría (8 x 3 = 24)",  matrizAuditoria.count() == 24)
)

comprobaciones.foreach { case (desc, ok) =>
  println(s"${if (ok) "✅ CORRECTO" else "❌ REVISAR"} - $desc")
}


VERIFICACIÓN FINAL - Caso de Estudio MediRed S.A.
✅ CORRECTO - Misión 3 - Ventas huérfanas detectadas
✅ CORRECTO - Misión 5 - Farmacias con doble inspección
✅ CORRECTO - Misión 7 - Farmacias solo en Q1
✅ CORRECTO - Misión 8 - Matriz auditoría (8 x 3 = 24)


import org.apache.spark.sql.functions.lit
prod: org.apache.spark.sql.package.DataFrame = [productoId: int, nombre: string ... 1 more field]
farm: org.apache.spark.sql.package.DataFrame = [farmaciaId: int, nombre: string]
vQ1: org.apache.spark.sql.package.DataFrame = [farmaciaId: int, productoId: int]
vQ2: org.apache.spark.sql.package.DataFrame = [farmaciaId: int, productoId: int]
vent: org.apache.spark.sql.package.DataFrame = [farmaciaId: int, productoId: int]
ventasHuerfanas: org.apache.spark.sql.package.DataFrame = [productoId: int, farmaciaId: int]
farmaciasAmbosTrims: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [farmaciaId: int]
soloQ1: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [farmaciaId: int]
matrizAuditoria: org.apache.spark.sql.package.DataFrame = [farmaciaId: int, nombre: string ... 1 more field]
comprobaciones: Seq[(String, Boolean)] = List(
  ("Misión 3 - Ventas huérfanas detectadas", true),
  ("Misión 5 - Farmacias con doble inspección", true),
